# Train a model, save it, use it

We train a straight-line regression once, save it to a file, and then use the saved model in an app.

The data here is **made up**, to show the steps. A real business would use its own sales records.

In [24]:
import numpy as np
import pandas as pd
import pickle
import sklearn

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [25]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
import pickle

## Made-up data

Two hundred days. Hotter days sell more ice cream, with some random noise.

In [26]:
np.random.seed(42)
n = 200

temperature = np.random.uniform(50, 100, n)
rain_probability = np.random.uniform(0, 100, n)

sales = (
    5 * temperature
    - 100
    - 1.2 * rain_probability
    + np.random.normal(0, 25, n)
)

df = pd.DataFrame({
    "temperature": temperature,
    "rain_probability": rain_probability,
    "sales": sales.clip(0)
})

df.head()

,temperature,rain_probability,sales
0,68.727006,64.203165,199.228202
1,97.535715,8.413996,378.106877
2,86.599697,16.162871,330.651864
3,79.932924,89.855419,184.081450
4,57.800932,60.642906,124.337332


In [27]:
px.scatter(df, x="temperature", y="sales", title="Cones sold versus daily high").show()

#Train and Test

In [28]:
X = df[["temperature", "rain_probability"]]
y = df["sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Temperature coefficient:", round(model.coef_[0], 2))
print("Rain probability coefficient:", round(model.coef_[1], 2))
print("MAE:", round(mean_absolute_error(y_test, predictions), 2))
print("R²:", round(r2_score(y_test, predictions), 2))

Temperature coefficient: 4.97
Rain probability coefficient: -1.21
MAE: 24.72
R²: 0.89


In [29]:
new_day = pd.DataFrame({
    "temperature": [85],
    "rain_probability": [20]
})

prediction = max(0, model.predict(new_day)[0])

print("Predicted sales:", round(prediction), "cones")

Predicted sales: 302 cones


## Fit the line

In [30]:
x = df[["temperature"]]
y = df["sales"]

model = LinearRegression()
model.fit(x, y)

print("Slope:", round(model.coef_[0], 2))
print("Intercept:", round(model.intercept_, 1))
print("R2:", round(model.score(x, y), 2))

Slope: 5.07
Intercept: -165.9
R2: 0.74


In [31]:
fig = px.scatter(df, x="temperature", y="sales", title="Cones sold versus daily high")
ends = pd.DataFrame({"temperature": [df["temperature"].min(), df["temperature"].max()]})
fig.add_scatter(x=ends["temperature"], y=model.predict(ends), mode="lines", name="Model")
fig.show()

Each extra degree adds about five cones. Now a prediction for a day with a high of 85 °F. The model wants a table with the same column name it was trained on.

In [32]:
new_day = pd.DataFrame({"temperature": [85]})
print(round(model.predict(new_day)[0]))

265


## Save it

`pickle` writes the trained model to a file. We train once and use the file as many times as we like.

In [33]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("requirements.txt", "w") as f:
    f.write(
        "streamlit\n"
        "pandas\n"
        "requests\n"
        f"scikit-learn=={sklearn.__version__}\n"
    )

## Load it back and check that it works

This is exactly what the app will do.

In [34]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print(round(loaded_model.predict(new_day)[0]))   # the same number as before

265


## Download it

The file has to go in your GitHub repository, next to `app.py`.

In [35]:
from google.colab import files

files.download("model.pkl")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
%%writefile app.py
import pickle
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests
import streamlit as st

st.set_page_config(
    page_title="Ice Cream Sales Predictor",
    page_icon="🍦"
)

CITIES = {
    "Pittsburgh": (40.44, -79.99),
    "New York": (40.71, -74.01),
    "Chicago": (41.88, -87.63),
    "Miami": (25.76, -80.19),
    "Los Angeles": (34.05, -118.24),
}

URL = "https://api.open-meteo.com/v1/forecast"

# Load the updated model trained on temperature and rain probability.
model_path = Path(__file__).parent / "model.pkl"

with open(model_path, "rb") as f:
    model = pickle.load(f)

st.title("🍦 Ice Cream Sales Predictor")

st.write(
    "Choose a city and date to estimate ice cream sales using "
    "the forecast temperature and chance of precipitation."
)

st.caption(
    "Classroom demo trained on simulated data. Incorrect predictions "
    "could lead to ordering too much or too little stock."
)

city = st.selectbox("City", list(CITIES))

selected_date = st.date_input(
    "Date",
    value=date.today() + timedelta(days=1),
    min_value=date.today(),
    max_value=date.today() + timedelta(days=6)
)

if st.button("Predict sales", type="primary"):
    latitude, longitude = CITIES[city]

    try:
        with st.spinner("Getting the weather forecast..."):
            response = requests.get(
                URL,
                params={
                    "latitude": latitude,
                    "longitude": longitude,
                    "daily": (
                        "temperature_2m_max,"
                        "precipitation_probability_max"
                    ),
                    "temperature_unit": "fahrenheit",
                    "timezone": "auto",
                    "start_date": str(selected_date),
                    "end_date": str(selected_date),
                },
                timeout=30,
            )

            response.raise_for_status()
            daily = response.json()["daily"]

            temperature = daily["temperature_2m_max"][0]
            rain_probability = daily["precipitation_probability_max"][0]

        if temperature is None or rain_probability is None:
            st.error(
                "The weather forecast is incomplete. "
                "Please choose another date."
            )
        else:
            # These columns must match the updated training notebook.
            features = pd.DataFrame({
                "temperature": [temperature],
                "rain_probability": [rain_probability],
            })

            prediction = max(0, model.predict(features)[0])

            col1, col2, col3 = st.columns(3)

            col1.metric(
                "Forecast high",
                f"{temperature:.0f} °F"
            )

            col2.metric(
                "Chance of precipitation",
                f"{rain_probability:.0f}%"
            )

            col3.metric(
                "Predicted sales",
                f"{prediction:.0f} cones"
            )

            if temperature < 50 or temperature > 100:
                st.warning(
                    "This temperature is outside the model's training "
                    "range of 50–100°F. Treat the estimate with caution."
                )

            if prediction > 300:
                st.info(
                    "Busy day expected: consider preparing more stock."
                )
            elif prediction > 200:
                st.info("A normal sales day is expected.")
            else:
                st.info(
                    "Slow day expected: consider preparing less stock."
                )

    except (
        requests.RequestException,
        ValueError,
        KeyError,
        IndexError,
        TypeError,
    ):
        st.error(
            "Unable to retrieve the forecast or calculate a prediction. "
            "Please try another date and confirm you uploaded the "
            "updated two-predictor model."
        )

st.caption("Weather forecasts provided by Open-Meteo.")
st.link_button("Weather data source", "https://open-meteo.com/")

Writing app.py


In [37]:
from google.colab import files

files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>